# Resultados del modelo de predicción de interrupciones

Notebook para analizar una ejecución de `etl/jobs/04_ml/job_ml_train.py`: comparación de los cinco modelos, curvas precision-recall, recall de interrupciones según el presupuesto de alertas, calibración, desglose por segmento, peso de las features y control de leakage.

Lee del lakehouse a través de Trino (`lakehouse.l3_gold`), nunca de extractos copiados a mano, así que el resultado se repite con el `run_id`. Requisitos en el portátil: el stack levantado (Trino en `localhost:8085`) y `pip install trino pandas matplotlib`.

Las decisiones de metodología (split temporal, muestra de negativos, corrección de la probabilidad, presupuesto de alertas, episodios) están en `docs/ml-training.md`.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import trino

conn = trino.dbapi.connect(host="localhost", port=8085, user="gridpredic", catalog="lakehouse", schema="l3_gold")


def q(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    return pd.DataFrame(rows, columns=[c[0] for c in cur.description])


pd.set_option("display.float_format", lambda v: f"{v:.5f}")
plt.rcParams["figure.figsize"] = (9, 4.5)

## 1. Ejecución

Por defecto la última de `ml_runs`. Para analizar otra, escribir su `run_id`.

In [ ]:
RUN_ID = None  # por ejemplo "ml_20261010T153000_ab12cd"

runs = q("SELECT run_id, creado_ts, dataset_version, etiqueta, splits_evaluados, artefactos_uri FROM ml_runs ORDER BY creado_ts DESC")
display(runs.head(10))

RUN_ID = RUN_ID or runs["run_id"].iloc[0]
run = q(f"SELECT * FROM ml_runs WHERE run_id = '{RUN_ID}'").iloc[0]
LABEL = run["etiqueta"]
SPLITS = run["splits_evaluados"].split(",")
print(RUN_ID, run["dataset_version"], LABEL, SPLITS)

## 2. Búsqueda de hiperparámetros

Todas las configuraciones probadas, con su PR-AUC en la muestra ponderada de valid. La elegida de cada modelo es la de mayor PR-AUC.

In [ ]:
search = pd.DataFrame(json.loads(run["busqueda_json"]))
search["params"] = search["params"].apply(lambda p: ", ".join(f"{k}={v}" for k, v in p.items() if k != "verbose"))
display(search[[c for c in ["modelo", "config", "params", "pr_auc_valid_muestra", "segundos_entrenamiento", "mejor_iteracion"] if c in search.columns]])

## 3. Comparación de modelos

Métricas sobre todas las filas del split. `lift_pr_auc` es cuántas veces mejor que puntuar al azar; con un 0,03 % de positivos la accuracy no dice nada y no se muestra. Las alertas son las N filas CT-hora con más puntuación de cada distribuidora y día.

In [ ]:
metrics = q(f"SELECT * FROM ml_metricas WHERE run_id = '{RUN_ID}'")
metrics["valor"] = metrics["valor"].astype(float)


def pivot(split, label=LABEL, segmento="total", columns=None):
    df = metrics[(metrics["split"] == split) & (metrics["etiqueta"] == label) & (metrics["segmento"] == segmento)]
    table = df.pivot_table(index="modelo", columns="metrica", values="valor")
    if columns:
        table = table[[c for c in columns if c in table.columns]]
    return table.sort_values(table.columns[0], ascending=False)


N = 10
MAIN = ["pr_auc", "lift_pr_auc", "roc_auc", f"precision@{N}", f"recall_filas@{N}", f"recall_episodios@{N}",
        f"antelacion_media_h@{N}", f"antelacion_mediana_h@{N}"]

for split in SPLITS:
    print(split)
    display(pivot(split, columns=MAIN))

In [ ]:
split = "valid"
table = pivot(split, columns=["pr_auc"])
prevalence = metrics[(metrics["split"] == split) & (metrics["etiqueta"] == LABEL) & (metrics["segmento"] == "total")
                     & (metrics["metrica"] == "prevalencia")]["valor"].iloc[0]

ax = table["pr_auc"].sort_values().plot.barh(color="#4C72B0")
ax.axvline(prevalence, color="grey", linestyle="--", label=f"tasa base {prevalence:.5f}")
ax.set_xlabel("PR-AUC (valid)")
ax.set_title("PR-AUC por modelo")
ax.legend()
plt.tight_layout()

## 4. Recall de interrupciones según el presupuesto de alertas

Para el operador lo que cuenta es cuántas interrupciones tuvieron al menos un aviso (recall por episodio) con el número de alertas que puede atender cada día.

In [ ]:
budgets = sorted({int(m.split("@")[1]) for m in metrics["metrica"] if m.startswith("recall_episodios@")})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, metric in zip(axes, ["recall_episodios", "precision"]):
    for model in pivot("valid").index:
        values = [pivot("valid").loc[model].get(f"{metric}@{b}", np.nan) for b in budgets]
        ax.plot(budgets, values, marker="o", label=model)
    ax.set_xlabel("alertas por día y distribuidora")
    ax.set_ylabel(metric)
    ax.set_xscale("log")
    ax.set_xticks(budgets, [str(b) for b in budgets])
axes[0].legend()
fig.suptitle("Valid: recall de interrupciones y precision según el presupuesto")
plt.tight_layout()

## 5. Curvas precision-recall

Sobre una muestra de las predicciones de valid: todos los positivos y el 2 % de los negativos (por hash, reproducible), con peso 50 en los negativos para que las curvas sean las de la población completa.

In [ ]:
models = [m for m in pivot("valid").index]
cols = ", ".join(f"p_{m}" for m in models)
sample = q(f'''
    SELECT {LABEL} AS y, {cols},
           CASE WHEN {LABEL} = 1 THEN 1.0 ELSE 50.0 END AS w
    FROM ml_predicciones
    WHERE run_id = '{RUN_ID}' AND split = 'valid'
      AND ({LABEL} = 1 OR abs(from_big_endian_64(xxhash64(to_utf8(ct_id || cast(hora AS varchar))))) % 100 < 2)
''')
print(len(sample), "filas,", int(sample["y"].sum()), "positivos")


def pr_curve(y, s, w):
    order = np.argsort(-s, kind="mergesort")
    y, w = y[order], w[order]
    tp = np.cumsum(w * y)
    fp = np.cumsum(w * (1 - y))
    return tp / tp[-1], tp / (tp + fp)


for model in models:
    if model == "tasa_base":
        continue
    recall, precision = pr_curve(sample["y"].to_numpy(float), sample[f"p_{model}"].to_numpy(float), sample["w"].to_numpy(float))
    plt.plot(recall, precision, label=model)
plt.axhline(prevalence, color="grey", linestyle="--", label="tasa base")
plt.yscale("log")
plt.xlabel("recall")
plt.ylabel("precision (log)")
plt.title("Curvas precision-recall en valid")
plt.legend()
plt.tight_layout()

## 6. Calibración de XGBoost

La probabilidad está corregida por el muestreo de negativos; aquí se comprueba si un 1 % predicho se corresponde con un 1 % observado. Bins por cuantiles de la probabilidad, sobre la misma muestra ponderada.

In [ ]:
if "p_xgboost" in sample.columns:
    df = sample[["y", "p_xgboost", "w"]].copy()
    df["bin"] = pd.qcut(df["p_xgboost"].rank(method="first"), 20, labels=False)
    cal = df.groupby("bin").apply(lambda g: pd.Series({
        "predicha": np.average(g["p_xgboost"], weights=g["w"]),
        "observada": np.average(g["y"], weights=g["w"]),
    }))
    plt.loglog(cal["predicha"], cal["observada"], marker="o", label="XGBoost")
    lims = [cal.min().min(), cal.max().max()]
    plt.plot(lims, lims, color="grey", linestyle="--", label="calibración perfecta")
    plt.xlabel("probabilidad predicha")
    plt.ylabel("frecuencia observada")
    plt.title("Calibración en valid")
    plt.legend()
    plt.tight_layout()

## 7. Por segmento

No se ocultan los segmentos débiles: PR-AUC y recall de interrupciones por distribuidora, tipo de zona del municipio (1 urbana a 4 rural dispersa) y CT con o sin telemetría propia.

In [ ]:
for segmento in ["distribuidora", "tipo_zona", "telemetria"]:
    df = metrics[(metrics["split"] == "valid") & (metrics["etiqueta"] == LABEL) & (metrics["segmento"] == segmento)
                 & metrics["metrica"].isin(["pr_auc", f"recall_episodios@{N}"])]
    table = df.pivot_table(index=["valor_segmento", "modelo"], columns="metrica", values="valor")
    counts = df.groupby("valor_segmento")[["n_filas", "n_positivos"]].first()
    print(segmento)
    display(counts)
    display(table.unstack("modelo").round(4))

## 8. Eventos locales frente a sistémicos

`y_1_3h_local` cuenta solo las interrupciones que afectan a un único CT. Si el modelo solo acierta "el día malo" (temporal, fallo aguas arriba), aquí cae.

In [ ]:
local = [c for c in metrics["etiqueta"].unique() if c.endswith("_local")]
for label in local:
    display(pivot("valid", label=label, columns=["pr_auc", "lift_pr_auc", f"recall_episodios@{N}", f"precision@{N}"]))

## 9. Peso de las features y control de leakage

Media del SHAP absoluto de XGBoost (TreeSHAP exacto de la librería) y PR-AUC de cada feature usada sola. Una feature que ordena los positivos casi perfectamente por sí sola es sospechosa de leakage y se revisa antes de aceptar el modelo.

In [ ]:
importance = q(f"SELECT modelo, feature, metrica, valor, alerta_leakage FROM ml_importancia WHERE run_id = '{RUN_ID}'")
importance["valor"] = importance["valor"].astype(float)

shap = importance[(importance["modelo"] == "xgboost") & (importance["metrica"] == "shap_medio_abs")]
top = shap.sort_values("valor", ascending=False).head(25).set_index("feature")["valor"]
top.sort_values().plot.barh(figsize=(8, 8), color="#4C72B0")
plt.xlabel("SHAP medio absoluto")
plt.title("Las 25 features con más peso en XGBoost")
plt.tight_layout()

In [ ]:
meta = q("SELECT feature, bloque, descripcion FROM feature_metadata")
blocks = shap.merge(meta, on="feature", how="left").groupby("bloque")["valor"].sum().sort_values(ascending=False)
display(blocks.to_frame("shap_total"))

uni = importance[importance["metrica"] == "pr_auc_univariante"].sort_values("valor", ascending=False)
print("Features con alerta de leakage:", uni[uni["alerta_leakage"]]["feature"].tolist() or "ninguna")
display(uni.head(15)[["feature", "valor", "alerta_leakage"]].merge(meta, on="feature", how="left"))

## 10. Test

Solo si la ejecución se lanzó con `--evaluar-test`. Test se mira una vez, con la configuración ya cerrada en valid. Las horas desde `evaluacion.test_hasta` no entran en la evaluación porque sus etiquetas están censuradas.

In [ ]:
if "test" in SPLITS:
    display(pivot("test", columns=MAIN))
    for label in local:
        display(pivot("test", label=label, columns=["pr_auc", f"recall_episodios@{N}"]))
else:
    print("Esta ejecución no evaluó test")